# 🎓 BITS Pilani — WILP | M.Tech (AIML)
## Machine Learning — Assignment 2: Classification Models & Cloud Deployment
---
### 👤 Student Information:
- **Student Name**: **PADAMATA ANURAG**
- **Roll Number**: **2026ak05613**
- **Email**: **2026ak05613@wilp.bits-pilani.ac.in**
- **Course**: Machine Learning (M.Tech AIML)
- **Execution Platform**: BITS Virtual Lab
---
### 📌 Project Outline & Objectives:
1. **Dataset**: Credit Card Fraud Detection Dataset 2026 (100,000 instances, 34 predictive features, binary target `is_fraud`).
2. **5 Pure NumPy Scratch Classifiers (No scikit-learn estimators)**:
   - Logistic Regression (Batch Gradient Descent with class weights)
   - Decision Tree Classifier (CART algorithm with Gini Impurity)
   - K-Nearest Neighbor Classifier (Batched Euclidean Distance)
   - Gaussian Naive Bayes Classifier (Continuous Gaussian Likelihoods)
   - Support Vector Machine (Linear SVM via Mini-Batch SGD with Hinge Loss)
3. **6 Evaluation Metrics (Calculated from Scratch)**:
   - Accuracy, AUC Score, Precision, Recall, F1 Score, Matthews Correlation Coefficient (MCC)
4. **Deployment Artifacts**:
   - Serialized pre-trained model artifacts stored in `model/`
   - Interactive web application in `app.py`

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Import scratch models and metrics
from models import (
    LogisticRegressionScratch,
    DecisionTreeScratch,
    KNNScratch,
    GaussianNaiveBayesScratch,
    SVMScratch,
)
from utils import MetricsScratch, Preprocessor

print("All libraries, pure NumPy scratch models, and utility modules loaded successfully!")

All libraries, pure NumPy scratch models, and utility modules loaded successfully!


## Step 1: Training Architecture & Preprocessing Pipeline

### Zero Data Leakage Protocol
To ensure rigorous scientific validity and strictly prevent data leakage:
1. The raw 100,000-sample dataset was split using a stratified 80/20 train-test partition (`train_evaluate.py`).
2. The `Preprocessor` (consisting of categorical label encoding, missing value imputation, and standard feature scaling) was fitted **exclusively on the training partition** (80,000 samples).
3. All five scratch models were trained on the training features (`X_train`) and serialized into `model/` for deployment.
4. Below is a demonstration of how each model architecture was configured and trained in `train_evaluate.py`.

In [2]:
# Model configuration demonstration as defined in train_evaluate.py
models_config = {
    'logistic_regression': LogisticRegressionScratch(
        learning_rate=0.1, n_iterations=500, threshold=0.3, class_weight='balanced'
    ),
    'decision_tree': DecisionTreeScratch(
        max_depth=12, min_samples_split=10, n_thresholds=20
    ),
    'knn': KNNScratch(k=7, batch_size=500),
    'naive_bayes': GaussianNaiveBayesScratch(var_smoothing=1e-9),
    'svm': SVMScratch(
        C=1.0, learning_rate=0.001, n_epochs=30, batch_size=512, class_weight='balanced'
    ),
}

for model_name, model_instance in models_config.items():
    print(f"Architecture ready: {model_name:<22} -> {model_instance.__class__.__name__}")

Architecture ready: logistic_regression    -> LogisticRegressionScratch
Architecture ready: decision_tree          -> DecisionTreeScratch
Architecture ready: knn                    -> KNNScratch
Architecture ready: naive_bayes            -> GaussianNaiveBayesScratch
Architecture ready: svm                    -> SVMScratch


## Step 2: Load Test Dataset & Verify Preprocessing
Loading `test_data.csv` (20,000 stratified instances across 34 continuous features and binary target `is_fraud`).

In [3]:
test_df = pd.read_csv('test_data.csv')
print(f"Test Data Shape: {test_df.shape[0]:,} rows x {test_df.shape[1]} columns")
print("Target Class Distribution:")
counts = test_df['is_fraud'].value_counts().to_dict()
print(f"  - Legitimate (0): {counts.get(0, 0):,} transactions ({counts.get(0, 0)/len(test_df)*100:.2f}%)")
print(f"  - Fraudulent (1): {counts.get(1, 0):,} transactions ({counts.get(1, 0)/len(test_df)*100:.2f}%)")

X_test = test_df.drop(columns=['is_fraud']).values
y_test = test_df['is_fraud'].values.astype(int)
assert X_test.shape[1] == 34, f"Expected 34 features, got {X_test.shape[1]}"

Test Data Shape: 20,000 rows x 35 columns
Target Class Distribution:
  - Legitimate (0): 19,589 transactions (97.95%)
  - Fraudulent (1): 411 transactions (2.05%)


## Step 3: Evaluate All 5 Saved Models on Test Partition
Loading trained `.joblib` models from `model/` and computing all 6 evaluation metrics dynamically locating the positive fraud class.

In [4]:
eval_models = [
    ("Logistic Regression", "logistic_regression.joblib"),
    ("Decision Tree", "decision_tree.joblib"),
    ("KNN (k=7)", "knn.joblib"),
    ("Naive Bayes (Gaussian)", "naive_bayes.joblib"),
    ("SVM (Linear SGD)", "svm.joblib"),
]

results = []
for name, filename in eval_models:
    model_path = os.path.join("model", filename)
    model = joblib.load(model_path)
    y_pred = model.predict(X_test)
    
    # Locate positive class dynamically based on fitted classes_ attribute
    classes = getattr(model, "classes_", None)
    if classes is None or 1 not in classes:
        raise ValueError(f"Model '{name}' missing fitted classes_ containing class 1")
    pos_idx = int(np.where(classes == 1)[0][0])
    y_proba = model.predict_proba(X_test)[:, pos_idx]
    
    scores = MetricsScratch.compute_all(y_test, y_pred, y_proba, positive_class=1)
    scores["Model"] = name
    results.append(scores)

results_df = pd.DataFrame(results)[["Model", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]]
print("\nOfficial Model Benchmark Results (20,000 Test Transactions):")
print("=" * 85)
print(results_df.to_string(index=False))
print("=" * 85)


Official Model Benchmark Results (20,000 Test Transactions):
                 Model  Accuracy      AUC  Precision   Recall       F1      MCC
   Logistic Regression   0.55875 0.835593   0.040821 0.909976 0.078136 0.131369
         Decision Tree   0.97505 0.650057   0.217949 0.082725 0.119929 0.123366
             KNN (k=7)   0.97925 0.620144   0.166667 0.002433 0.004796 0.017841
Naive Bayes (Gaussian)   0.90660 0.819320   0.100384 0.445255 0.163832 0.178206
      SVM (Linear SGD)   0.66745 0.825493   0.049913 0.841849 0.094239 0.150742


## Step 4: Comparative Analysis & Insights

### Key Observations Across the 5 Models:
1. **Gaussian Naive Bayes (Overall Winner)**:
   - Achieves the highest **F1 Score (0.1638)** and highest **MCC Score (0.1782)** while maintaining **90.66% Accuracy**, **0.8193 AUC**, and **44.53% Recall**.
   - By independently estimating class-conditional Gaussian densities, it reliably flags fraud attacks without majority-class bias.

2. **Logistic Regression**:
   - Demonstrates the strongest overall discriminatory power with the highest **AUC of 0.8356** and exceptional **Recall of 91.00%** (detecting 374 of 411 fraud cases).
   - Serves as an ideal high-sensitivity frontline triage filter where detecting fraud takes priority over false alarms.

3. **Linear Support Vector Machine (SGD)**:
   - Reaches strong discrimination with an **AUC of 0.8255** and **Recall of 84.18%** (identifying 346 fraud transactions) alongside **66.75% Accuracy**.
   - Class-weighted hinge loss successfully counters the 47:1 imbalance, mirroring Logistic Regression's high-sensitivity profile.

4. **Decision Tree (CART)**:
   - Attains **97.51% Accuracy** and **0.6501 AUC** with **8.27% Recall** and **21.79% Precision** (F1: 0.1199, MCC: 0.1234).
   - Orthogonal axis-aligned splits effectively capture non-linear transaction spending thresholds while maintaining high overall accuracy.

5. **K-Nearest Neighbors (k=7)**:
   - Delivers the highest nominal **Accuracy (97.92%)** and a **Precision of 16.67%**, but suffers severely from the curse of dimensionality across 34 continuous normalized features under extreme class imbalance, leading to low **Recall (0.24%)** and **F1 (0.0048)**.

### Conclusion:
Under extreme 47:1 class imbalance, raw accuracy is deeply misleading. **Gaussian Naive Bayes** represents the optimal operational classifier for credit card fraud detection by striking the best harmonic balance between fraud sensitivity (Recall) and precision.